In [ ]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [ ]:
%run ../common/enviroment_config

In [ ]:
%run ../common/gold_helper

In [ ]:
target_table = f"{catalog_name}.{gold_schema}.dim_constructors"

In [ ]:
from pyspark.sql import functions as F

In [ ]:
constructors_df = (
    spark.table(f"{catalog_name}.{silver_schema}.constructors")
         .filter(F.col("batch_id") == v_batch_id)
)

In [ ]:
ref_nationality_region_df = spark.table(f"{catalog_name}.{gold_schema}.ref_nationality_region")

In [ ]:
dim_constructors_df = (
    constructors_df
        .join(
            ref_nationality_region_df,
            constructors_df.nationality == ref_nationality_region_df.nationality,
            "left"
            )
        .select (
            constructors_df.constructor_id,
            constructors_df.constructor_name,
            constructors_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)
display(dim_constructors_df)

In [ ]:
write_to_gold(
    input_df=dim_constructors_df,
    target_table=target_table,
    merge_condition="t.constructor_id = s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "nationality_region"
    ]
)

# COMMAND ----------

display(spark.table(target_table))